<a href="https://colab.research.google.com/github/Nasim-Amani/solar/blob/main/cleaned_solar_data_Oct26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

from pathlib import Path
import pandas as pd

# 1. Locate Excel files in Google Colab
DATA_DIR = Path("/content")
files = sorted(DATA_DIR.glob("*.xlsx"))

if not files:
    raise FileNotFoundError(
        f"No .xlsx files found in {DATA_DIR}. "
        "Upload your Excel files or mount Google Drive."
    )

# 2. Define the expected measurement schema
expected_columns = [
    "ID", "Date", "Time", "Amb Tempr", "Solar Rad",
    "Tempr PV 1", "Tempr PV 2", "Tempr PV 3",
    "Parameter 8", "PV 1 SA V", "PV 2 GT V",
    "PV 3 Tr V", "PV 4 Hor V", "PV 1 SA I",
    "PV 2 GT I", "PV 3 Tr I", "PV 4 Hor I"
]

# 3. Inspect each file and load its data
dataframes = []
reference_columns = None
audit = []

for file in files:
    df = pd.read_excel(file)
    columns = df.columns.tolist()

    if reference_columns is None:
        reference_columns = columns

    audit.append({
        "file": file.name,
        "rows": len(df),
        "columns": len(columns),
        "missing_cells": int(df.isna().sum().sum()),
        "schema_matches_first_file": (
            set(columns) == set(reference_columns)
        ),
        "missing_expected_columns": [
            c for c in expected_columns if c not in columns
        ],
        "unexpected_columns": [
            c for c in columns if c not in expected_columns
        ],
    })

    # Preserve source-file provenance
    df["source_file"] = file.name
    dataframes.append(df)

# 4. Display the audit report
audit_df = pd.DataFrame(audit)
display(audit_df)

# 5. Combine files only after reporting schema differences
final_df = pd.concat(dataframes, ignore_index=True, sort=False)

print(f"Files processed: {len(files)}")
print(f"Combined shape: {final_df.shape}")
print(f"Duplicate column labels: "
      f"{final_df.columns.duplicated().any()}")
display(final_df.head())


,file,rows,columns,missing_cells,schema_matches_first_file,missing_expected_columns,unexpected_columns
0,1.xlsx,60000,17,0,True,[],[]
1,10.xlsx,60000,17,0,True,[],[]
2,11.xlsx,60000,17,0,True,[],[]
3,12.xlsx,60000,17,0,True,[],[]
4,13.xlsx,60000,17,0,True,[],[]
5,14.xlsx,60000,17,0,True,[],[]
6,15.xlsx,60000,17,0,True,[],[]
7,16.xlsx,60000,17,0,True,[],[]
8,17.xlsx,56007,17,0,True,[],[]
9,2.xlsx,60000,17,0,True,[],[]


Files processed: 17
Combined shape: (1016008, 18)
Duplicate column labels: False


,ID,Date,Time,Amb Tempr,Solar Rad,Tempr PV 1,Tempr PV 2,Tempr PV 3,Parameter 8,PV 1 SA V,PV 2 GT V,PV 3 Tr V,PV 4 Hor V,PV 1 SA I,PV 2 GT I,PV 3 Tr I,PV 4 Hor I,source_file
0,1,2013-02-15,18:49:44,20.922115,2.437606,13.257263,12.494228,13.441733,0,0.0,0,0.0,-0.617625,-0.013561,-0.009259,0.000398,-0.086931,1.xlsx
1,2,2013-02-15,18:49:45,20.897755,2.468246,13.254468,12.499819,13.450118,0,0.0,0,0.0,-0.617625,-0.013561,-0.008453,0.001171,-0.084999,1.xlsx
2,3,2013-02-15,18:49:46,20.897755,2.468246,13.254468,12.499819,13.450118,0,0.0,0,0.0,-0.617625,-0.013561,-0.008453,0.001171,-0.084999,1.xlsx
3,4,2013-02-15,18:49:48,20.922115,2.560166,13.254468,12.505408,13.458504,0,0.0,0,0.0,-0.617625,-0.013561,-0.007647,0.001944,-0.083453,1.xlsx
4,5,2013-02-15,18:49:48,20.934296,2.652086,13.262854,12.505408,13.458504,0,0.0,0,0.0,-0.617625,-0.011653,-0.007244,0.003103,-0.083839,1.xlsx


In [4]:

import pandas as pd

# 1. Inspect how Excel time values were imported
print("Date dtype:", final_df["Date"].dtype)
print("Time dtype:", final_df["Time"].dtype)
print("Sample time values:")
print(final_df["Time"].head(10).map(repr).to_list())

# 2. Parse the date column
dates = pd.to_datetime(final_df["Date"], errors="coerce", dayfirst=True)

# 3. Report potentially invalid date/time values before combining
print("\nInvalid dates:", int(dates.isna().sum()))
print("Missing times:", int(final_df["Time"].isna().sum()))

# 4. Combine only after checking the imported time representation
# This conversion works when Time contains recognizable time strings
# or datetime-like values, including Excel-imported midnight values.
times = pd.to_datetime(
    final_df["Time"].astype(str),
    errors="coerce"
)

# Extract the time-of-day; ignore any artificial date attached to Time
time_delta = (
    pd.to_timedelta(times.dt.hour, unit="h")
    + pd.to_timedelta(times.dt.minute, unit="m")
    + pd.to_timedelta(times.dt.second, unit="s")
)

# Keep fractional seconds where present
time_delta += pd.to_timedelta(times.dt.microsecond, unit="us")

# 5. Construct DateTime without modifying the original columns
final_df["DateTime"] = dates.dt.normalize() + time_delta

# Do not silently accept failed conversions
invalid_datetime = final_df["DateTime"].isna()
print("Invalid combined timestamps:", int(invalid_datetime.sum()))

# 6. Check timestamp duplicates before deciding what they mean
duplicate_mask = final_df.duplicated(
    subset=["DateTime"], keep=False
)

print("Rows sharing a timestamp:", int(duplicate_mask.sum()))
print("Unique timestamps:", final_df["DateTime"].nunique())

# 7. Sort chronologically, keeping original columns for traceability
final_df = final_df.sort_values(
    ["DateTime", "ID"], na_position="last"
).reset_index(drop=True)

# 8. Move DateTime to the first column
final_df = final_df[
    ["DateTime"] + [c for c in final_df.columns if c != "DateTime"]
]

print("Chronologically sorted:", final_df["DateTime"].is_monotonic_increasing)
display(final_df.head())


Date dtype: datetime64[ns]
Time dtype: object
Sample time values:
['datetime.time(18, 49, 44)', 'datetime.time(18, 49, 45)', 'datetime.time(18, 49, 46)', 'datetime.time(18, 49, 48)', 'datetime.time(18, 49, 48)', 'datetime.time(18, 49, 49)', 'datetime.time(18, 49, 50)', 'datetime.time(18, 49, 51)', 'datetime.time(18, 49, 52)', 'datetime.time(18, 49, 53)']

Invalid dates: 0
Missing times: 0


/tmp/ipykernel_881/163689220.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  times = pd.to_datetime(


Invalid combined timestamps: 0
Rows sharing a timestamp: 184010
Unique timestamps: 924003
Chronologically sorted: True


,DateTime,ID,Date,Time,Amb Tempr,Solar Rad,Tempr PV 1,Tempr PV 2,Tempr PV 3,Parameter 8,PV 1 SA V,PV 2 GT V,PV 3 Tr V,PV 4 Hor V,PV 1 SA I,PV 2 GT I,PV 3 Tr I,PV 4 Hor I,source_file
0,2013-02-15 18:49:44,1,2013-02-15,18:49:44,20.922115,2.437606,13.257263,12.494228,13.441733,0,0.0,0,0.0,-0.617625,-0.013561,-0.009259,0.000398,-0.086931,1.xlsx
1,2013-02-15 18:49:45,2,2013-02-15,18:49:45,20.897755,2.468246,13.254468,12.499819,13.450118,0,0.0,0,0.0,-0.617625,-0.013561,-0.008453,0.001171,-0.084999,1.xlsx
2,2013-02-15 18:49:46,3,2013-02-15,18:49:46,20.897755,2.468246,13.254468,12.499819,13.450118,0,0.0,0,0.0,-0.617625,-0.013561,-0.008453,0.001171,-0.084999,1.xlsx
3,2013-02-15 18:49:48,4,2013-02-15,18:49:48,20.922115,2.560166,13.254468,12.505408,13.458504,0,0.0,0,0.0,-0.617625,-0.013561,-0.007647,0.001944,-0.083453,1.xlsx
4,2013-02-15 18:49:48,5,2013-02-15,18:49:48,20.934296,2.652086,13.262854,12.505408,13.458504,0,0.0,0,0.0,-0.617625,-0.011653,-0.007244,0.003103,-0.083839,1.xlsx


In [5]:

# Remove duplicate measurements before timestamp correction

measurement_cols = [
    "Amb Tempr", "Solar Rad",
    "Tempr PV 1", "Tempr PV 2", "Tempr PV 3",
    "Parameter 8", "PV 1 SA V", "PV 2 GT V",
    "PV 3 Tr V", "PV 4 Hor V", "PV 1 SA I",
    "PV 2 GT I", "PV 3 Tr I", "PV 4 Hor I"
]

# Compare the original timestamp and all sensor measurements
duplicate_cols = ["DateTime"] + measurement_cols

# Preserve removed rows for traceability
duplicate_mask = final_df.duplicated(
    subset=duplicate_cols,
    keep="first"
)

removed_duplicates = final_df.loc[duplicate_mask].copy()
final_df = final_df.loc[~duplicate_mask].copy()
final_df = final_df.reset_index(drop=True)

print(f"Duplicate records removed: {len(removed_duplicates):,}")
print(f"Records remaining: {len(final_df):,}")

# Verify that no matching duplicate groups remain
remaining = final_df.duplicated(
    subset=duplicate_cols,
    keep=False
)

print(f"Rows in remaining duplicate groups: {remaining.sum():,}")

display(removed_duplicates)


Duplicate records removed: 17
Records remaining: 1,015,991
Rows in remaining duplicate groups: 0


,DateTime,ID,Date,Time,Amb Tempr,Solar Rad,Tempr PV 1,Tempr PV 2,Tempr PV 3,Parameter 8,PV 1 SA V,PV 2 GT V,PV 3 Tr V,PV 4 Hor V,PV 1 SA I,PV 2 GT I,PV 3 Tr I,PV 4 Hor I,source_file
38928,2013-02-16 05:38:32,38929,2013-02-16,05:38:32,15.975005,2.774646,8.754519,8.094898,9.472834,0,0.000000,0,0.000000,-0.617625,-0.030726,-0.007244,0.006967,-0.037857,1.xlsx
100984,2013-02-16 22:52:48,100985,2013-02-16,22:52:48,20.004555,2.774646,12.709443,12.072184,12.902298,0,0.000000,0,0.000000,-0.617625,-0.040263,-0.010065,0.018559,-0.025105,2.xlsx
114032,2013-02-17 02:30:16,114033,2013-02-17,02:30:16,18.382586,2.805286,11.314738,10.266613,11.672499,0,0.000000,0,0.000000,-0.617625,-0.044077,-0.008856,0.019332,-0.023946,2.xlsx
120000,2013-02-17 04:09:43,120000,2013-02-17,04:09:43,17.601034,2.805286,10.031834,9.333083,10.593628,0,0.000000,0,0.000000,-0.617625,-0.045985,-0.008856,0.020105,-0.026264,3.xlsx
212273,2013-02-18 05:47:36,212273,2013-02-18,05:47:36,20.327326,2.590806,13.234903,12.620004,13.662539,0,0.000000,0,0.000000,-0.617625,-0.074594,0.008071,0.021264,-0.028196,4.xlsx
236149,2013-02-18 12:25:31,236149,2013-02-18,12:25:31,27.962154,842.126831,53.770790,53.133530,48.862770,0,38.695862,0,0.000000,18.526323,-0.078409,6.659348,0.026287,0.071497,4.xlsx
261841,2013-02-18 19:33:44,261841,2013-02-18,19:33:44,24.998356,2.652086,17.304424,16.521824,17.452559,0,0.000000,0,0.000000,-0.617625,-0.076501,0.029027,0.022810,-0.028583,5.xlsx
273481,2013-02-18 22:47:45,273481,2013-02-18,22:47:45,22.414165,2.805286,14.394829,13.542354,14.819669,0,0.000000,0,0.000000,-0.617625,-0.078409,0.029430,0.022037,-0.029356,5.xlsx
410957,2013-02-20 12:59:00,410957,2013-02-20,12:59:00,24.395445,396.222900,37.685566,36.128750,33.261078,0,40.789402,0,0.000000,18.069666,-0.095574,4.607624,0.013922,0.035948,7.xlsx
608109,2013-02-22 19:44:51,608109,2013-02-22,19:44:51,23.191654,2.682726,14.615633,13.489248,15.199788,0,0.000000,0,0.000000,-0.617625,-0.108926,0.043132,-0.001920,0.032084,11.xlsx


In [6]:

# Correct repeated timestamps after duplicate removal

data_copy = final_df.copy()

# Preserve original timestamps before reconstruction
data_copy["DateTime_original"] = data_copy["DateTime"]

# Sort records chronologically
data_copy = data_copy.sort_values(
    ["DateTime", "ID"]
).reset_index(drop=True)

# Reconstruct a strictly increasing timeline
valid_mask = data_copy["DateTime"].notna()
original_times = data_copy.loc[valid_mask, "DateTime"].tolist()

corrected_times = []

for timestamp in original_times:
    if not corrected_times:
        corrected_times.append(timestamp)
    else:
        corrected_times.append(
            max(
                timestamp,
                corrected_times[-1] + pd.Timedelta(seconds=1)
            )
        )

data_copy.loc[valid_mask, "DateTime"] = corrected_times

# Verify the corrected timestamps
valid_times = data_copy.loc[valid_mask, "DateTime"]

print(f"Records processed: {len(data_copy):,}")
print("Remaining duplicate timestamps:",
      int(valid_times.duplicated(keep=False).sum()))
print("Chronologically sorted:",
      valid_times.is_monotonic_increasing)

final_df = data_copy
display(final_df.head())


Records processed: 1,015,991
Remaining duplicate timestamps: 0
Chronologically sorted: True


,DateTime,ID,Date,Time,Amb Tempr,Solar Rad,Tempr PV 1,Tempr PV 2,Tempr PV 3,Parameter 8,PV 1 SA V,PV 2 GT V,PV 3 Tr V,PV 4 Hor V,PV 1 SA I,PV 2 GT I,PV 3 Tr I,PV 4 Hor I,source_file,DateTime_original
0,2013-02-15 18:49:44,1,2013-02-15,18:49:44,20.922115,2.437606,13.257263,12.494228,13.441733,0,0.0,0,0.0,-0.617625,-0.013561,-0.009259,0.000398,-0.086931,1.xlsx,2013-02-15 18:49:44
1,2013-02-15 18:49:45,2,2013-02-15,18:49:45,20.897755,2.468246,13.254468,12.499819,13.450118,0,0.0,0,0.0,-0.617625,-0.013561,-0.008453,0.001171,-0.084999,1.xlsx,2013-02-15 18:49:45
2,2013-02-15 18:49:46,3,2013-02-15,18:49:46,20.897755,2.468246,13.254468,12.499819,13.450118,0,0.0,0,0.0,-0.617625,-0.013561,-0.008453,0.001171,-0.084999,1.xlsx,2013-02-15 18:49:46
3,2013-02-15 18:49:48,4,2013-02-15,18:49:48,20.922115,2.560166,13.254468,12.505408,13.458504,0,0.0,0,0.0,-0.617625,-0.013561,-0.007647,0.001944,-0.083453,1.xlsx,2013-02-15 18:49:48
4,2013-02-15 18:49:49,5,2013-02-15,18:49:48,20.934296,2.652086,13.262854,12.505408,13.458504,0,0.0,0,0.0,-0.617625,-0.011653,-0.007244,0.003103,-0.083839,1.xlsx,2013-02-15 18:49:48


### Timestamp Correction and Validation

1. **Copy the original timestamps:** The original values are saved in `DateTime_original` so they can be checked later.
2. **Sort the records:** The data is sorted by `DateTime` and `ID`.
3. **Correct repeated timestamps:** Each timestamp is compared with the previous corrected timestamp. If it is less than one second later, it is moved forward to ensure at least a one-second interval.
4. **Check the results:** The code checks whether any duplicate timestamps remain and whether the records are sorted in chronological order.

**Result:** All 1,015,991 records have timestamps, no duplicate timestamps remain, and the records are sorted chronologically.

**Note:** The corrected timestamps are reconstructed values. The original timestamps remain available in `DateTime_original`.

In [7]:

# Final data-quality check

# 1. Check missing values
print("Missing values per column:")
display(final_df.isna().sum().to_frame("Missing values"))

# 2. Check exact repeated measurements (do not delete them)
measurement_cols = [
    c for c in expected_columns
    if c not in ["ID", "Date", "Time"]
]

duplicate_mask = final_df.duplicated(
    subset=measurement_cols + ["DateTime_original"],
    keep=False
)

print(f"Rows with identical measurements and original timestamps: "
      f"{duplicate_mask.sum():,}")

# 3. Check numeric measurement columns
numeric_cols = measurement_cols

for col in numeric_cols:
    final_df[col] = pd.to_numeric(final_df[col], errors="coerce")

print("\nNumeric conversion / missing-value check:")
display(final_df[numeric_cols].isna().sum().to_frame("Missing or non-numeric"))

# 4. Flag basic potential measurement issues
quality_flags = pd.DataFrame(index=final_df.index)

quality_flags["Negative solar radiation"] = final_df["Solar Rad"] < 0

temperature_cols = [
    "Amb Tempr", "Tempr PV 1", "Tempr PV 2", "Tempr PV 3"
]

for col in temperature_cols:
    quality_flags[f"{col} outside -50 to 100 °C"] = (
        (final_df[col] < -50) | (final_df[col] > 100)
    )

print("\nPotential quality issues (flagged rows per check):")
display(quality_flags.sum().astype(int).to_frame("Flagged rows"))

print(f"\nFinal dataset size: {len(final_df):,} rows × {len(final_df.columns)} columns")


Missing values per column:


,Missing values
DateTime,0
ID,0
Date,0
Time,0
Amb Tempr,0
Solar Rad,0
Tempr PV 1,0
Tempr PV 2,0
Tempr PV 3,0
Parameter 8,0


Rows with identical measurements and original timestamps: 0

Numeric conversion / missing-value check:


,Missing or non-numeric
Amb Tempr,0
Solar Rad,0
Tempr PV 1,0
Tempr PV 2,0
Tempr PV 3,0
Parameter 8,0
PV 1 SA V,0
PV 2 GT V,0
PV 3 Tr V,0
PV 4 Hor V,0



Potential quality issues (flagged rows per check):


,Flagged rows
Negative solar radiation,0
Amb Tempr outside -50 to 100 °C,0
Tempr PV 1 outside -50 to 100 °C,0
Tempr PV 2 outside -50 to 100 °C,0
Tempr PV 3 outside -50 to 100 °C,0



Final dataset size: 1,015,991 rows × 20 columns


In [9]:
# Check zero values in selected measurement columns

cols = [
    "Amb Tempr", "Solar Rad",
    "Tempr PV 1", "Tempr PV 2", "Tempr PV 3",
    "Parameter 8",
    "PV 1 SA V", "PV 2 GT V", "PV 3 Tr V", "PV 4 Hor V",
    "PV 1 SA I", "PV 2 GT I", "PV 3 Tr I", "PV 4 Hor I"
]
zero_counts = final_df[cols].eq(0).sum()

zero_summary = pd.DataFrame({
    "Zero Count": zero_counts,
    "Zero Percentage (%)": (zero_counts / len(final_df) * 100).round(2)
})

display(zero_summary)

,Zero Count,Zero Percentage (%)
Amb Tempr,0,0.00
Solar Rad,0,0.00
Tempr PV 1,0,0.00
Tempr PV 2,0,0.00
Tempr PV 3,0,0.00
Parameter 8,1015991,100.00
PV 1 SA V,478792,47.13
PV 2 GT V,1015991,100.00
PV 3 Tr V,1007168,99.13
PV 4 Hor V,0,0.00


In [11]:
# Remove columns used only for data preparation and tracking

cols_to_drop = [
    "Date",
    "Time",
    "DateTime_original",
    "source_file",
    "Parameter 8",
    "PV 2 GT V",
    "PV 3 Tr V",
    "ID"
]

# Drop only columns that still exist
final_df = final_df.drop(
    columns=[col for col in cols_to_drop if col in final_df.columns]
)

print("Updated dataset shape:", final_df.shape)
print("Remaining columns:")
print(final_df.columns.tolist())

display(final_df.head())

Updated dataset shape: (1015991, 12)
Remaining columns:
['DateTime', 'Amb Tempr', 'Solar Rad', 'Tempr PV 1', 'Tempr PV 2', 'Tempr PV 3', 'PV 1 SA V', 'PV 4 Hor V', 'PV 1 SA I', 'PV 2 GT I', 'PV 3 Tr I', 'PV 4 Hor I']


,DateTime,Amb Tempr,Solar Rad,Tempr PV 1,Tempr PV 2,Tempr PV 3,PV 1 SA V,PV 4 Hor V,PV 1 SA I,PV 2 GT I,PV 3 Tr I,PV 4 Hor I
0,2013-02-15 18:49:44,20.922115,2.437606,13.257263,12.494228,13.441733,0.0,-0.617625,-0.013561,-0.009259,0.000398,-0.086931
1,2013-02-15 18:49:45,20.897755,2.468246,13.254468,12.499819,13.450118,0.0,-0.617625,-0.013561,-0.008453,0.001171,-0.084999
2,2013-02-15 18:49:46,20.897755,2.468246,13.254468,12.499819,13.450118,0.0,-0.617625,-0.013561,-0.008453,0.001171,-0.084999
3,2013-02-15 18:49:48,20.922115,2.560166,13.254468,12.505408,13.458504,0.0,-0.617625,-0.013561,-0.007647,0.001944,-0.083453
4,2013-02-15 18:49:49,20.934296,2.652086,13.262854,12.505408,13.458504,0.0,-0.617625,-0.011653,-0.007244,0.003103,-0.083839


## Zero-Value Analysis and Column Selection

- **Parameter 8 (100% zeros):**  remove.
- **PV 2 GT V (100% zeros):**  remove.
- **PV 3 Tr V (99.13% zeros):**  remove.
- **PV 1 SA V (47.13% zeros):** Check whether zeros occur mainly at night or also during daylight before deciding.

**PV 4 is without temperature data**

In [12]:
# Rename columns for consistent, code-friendly naming

final_df = final_df.rename(columns={
    "DateTime": "DateTime",
    "Amb Tempr": "Amb_Tempr",
    "Solar Rad": "Solar_Rad",
    "Tempr PV 1": "Tempr_PV_1",
    "Tempr PV 2": "Tempr_PV_2",
    "Tempr PV 3": "Tempr_PV_3",
    "PV 1 SA V": "PV_1_SA_V",
    "PV 4 Hor V": "PV_4_Hor_V",
    "PV 1 SA I": "PV_1_SA_I",
    "PV 2 GT I": "PV_2_GT_I",
    "PV 3 Tr I": "PV_3_Tr_I",
    "PV 4 Hor I": "PV_4_Hor_I"
})

print("Updated column names:")
print(final_df.columns.tolist())

display(final_df.head())

Updated column names:
['DateTime', 'Amb_Tempr', 'Solar_Rad', 'Tempr_PV_1', 'Tempr_PV_2', 'Tempr_PV_3', 'PV_1_SA_V', 'PV_4_Hor_V', 'PV_1_SA_I', 'PV_2_GT_I', 'PV_3_Tr_I', 'PV_4_Hor_I']


,DateTime,Amb_Tempr,Solar_Rad,Tempr_PV_1,Tempr_PV_2,Tempr_PV_3,PV_1_SA_V,PV_4_Hor_V,PV_1_SA_I,PV_2_GT_I,PV_3_Tr_I,PV_4_Hor_I
0,2013-02-15 18:49:44,20.922115,2.437606,13.257263,12.494228,13.441733,0.0,-0.617625,-0.013561,-0.009259,0.000398,-0.086931
1,2013-02-15 18:49:45,20.897755,2.468246,13.254468,12.499819,13.450118,0.0,-0.617625,-0.013561,-0.008453,0.001171,-0.084999
2,2013-02-15 18:49:46,20.897755,2.468246,13.254468,12.499819,13.450118,0.0,-0.617625,-0.013561,-0.008453,0.001171,-0.084999
3,2013-02-15 18:49:48,20.922115,2.560166,13.254468,12.505408,13.458504,0.0,-0.617625,-0.013561,-0.007647,0.001944,-0.083453
4,2013-02-15 18:49:49,20.934296,2.652086,13.262854,12.505408,13.458504,0.0,-0.617625,-0.011653,-0.007244,0.003103,-0.083839


In [13]:
# Save the cleaned dataset as a CSV file

output_path = "/content/cleaned_solar_data_Oct26.csv"
final_df.to_csv(output_path, index=False)

print(f"Dataset saved to: {output_path}")

Dataset saved to: /content/cleaned_solar_data_Oct26.csv


In [14]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1015991 entries, 0 to 1015990
Data columns (total 12 columns):
 #   Column      Non-Null Count    Dtype         
---  ------      --------------    -----         
 0   DateTime    1015991 non-null  datetime64[ns]
 1   Amb_Tempr   1015991 non-null  float64       
 2   Solar_Rad   1015991 non-null  float64       
 3   Tempr_PV_1  1015991 non-null  float64       
 4   Tempr_PV_2  1015991 non-null  float64       
 5   Tempr_PV_3  1015991 non-null  float64       
 6   PV_1_SA_V   1015991 non-null  float64       
 7   PV_4_Hor_V  1015991 non-null  float64       
 8   PV_1_SA_I   1015991 non-null  float64       
 9   PV_2_GT_I   1015991 non-null  float64       
 10  PV_3_Tr_I   1015991 non-null  float64       
 11  PV_4_Hor_I  1015991 non-null  float64       
dtypes: datetime64[ns](1), float64(11)
memory usage: 93.0 MB
